# ⚽ HaxBall AI: Treinador de Reforço PPO de Alta Performance & Self-Play League

Este notebook executa o pipeline de **Reinforcement Learning (PPO)** em escala industrial:
1. **Ambiente Vetorizado 61D:** Percepção espacial multi-agente normalizada.
2. **Recompensas Sem Distorção:** Bônus de aproximação, condução ofensiva, cobertura defensiva de gol e penalidade de chute no ar (*whiff penalty*).
3. **Liga de Self-Play Histórica:** O agente treina contra clones recentes e checkpoints históricos para evitar ciclos de esquecimento (*Rock-Paper-Scissors*).
4. **Telemetria e Gráficos ao Vivo:** Recompensa, Win-Rate, Perdas e Entropia.
5. **Copa HaxBall (Grand Tournament Final):** Confronto Round-Robin entre gerações da IA e bateria de testes contra os 8 bots arquétipos!

In [ ]:
# 1. Configuração do Hardware e GPU
import os, sys, time, math, random
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ PyTorch Version: {torch.__version__}')
print(f'✓ Dispositivo de Execução: {device}')
if torch.cuda.is_available():
    print(f'✓ GPU Ativa: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Clonar repositório e instalar dependências
!git clone https://github.com/CaioHVectorA/reinforcement-cases.git haxball_repo 2>/dev/null || (cd haxball_repo && git pull origin main)
%cd haxball_repo
!pip install -q pygame gymnasium torch torchvision matplotlib tabulate


In [ ]:
# @title 4. ⚙️ Hiperparâmetros do Treinador PPO & Self-Play { run: "auto" }
TOTAL_ITERACOES = 1000 #@param {type:"slider", min:100, max:10000, step:100}
MAPA_ESTADIO = "futsal_2v2" #@param ["futsal_2v2", "classic", "big_stadium", "micro_1v1", "small_classic", "futsal_3v3", "dodgeball"]
ESCALA_DO_MAPA = 1.0 #@param {type:"slider", min:0.5, max:3.0, step:0.25}
PASSOS_POR_ITERAÇÃO = 4096 #@param [2048, 4096, 8192]
LEARNING_RATE = 0.0003 #@param {type:"number"}
ENTROPY_COEF = 0.012 #@param {type:"number"}
WARM_START = False #@param {type:"boolean"}

class PPOConfig:
    stadium_file = f"haxball/maps/{MAPA_ESTADIO}.hbs"
    stadium_scale = float(ESCALA_DO_MAPA)
    policy_type = "mlp"  # "mlp" (128x128) ou "attention"
    obs_dim = 61         # Percepção periférica completa (Ego, Bola, 4 Aliados, 5 Inimigos)
    num_actions = 18     # 9 direções x 2 estados de chute
    
    lr = LEARNING_RATE
    gamma = 0.995
    gae_lambda = 0.95
    clip_coef = 0.2
    ent_coef = ENTROPY_COEF
    vf_coef = 0.5
    max_grad_norm = 0.5
    num_steps = int(PASSOS_POR_ITERAÇÃO)
    batch_size = 128
    update_epochs = 8
    
    total_iterations = int(TOTAL_ITERACOES)
    stage1_iterations = max(50, int(TOTAL_ITERACOES * 0.20)) # 20% inicial contra oponente base
    plot_frequency = 5 # Atualiza gráficos a cada 5 iterações
    eval_frequency = 25 # Gauntlet a cada 25 iterações
    checkpoint_dir = "checkpoints"

cfg = PPOConfig()
os.makedirs(cfg.checkpoint_dir, exist_ok=True)
total_phys_steps = cfg.total_iterations * cfg.num_steps
print(f"✓ Configuração PPO pronta! Total de passos a simular: {total_phys_steps:,}")
print(f"✓ Estádio: {MAPA_ESTADIO} | Escala de Dimensão: {cfg.stadium_scale:.2f}x")
print(f"✓ Fase 1 (Fundamentos): Iter 1 a {cfg.stage1_iterations} | Fase 2 (Liga de Self-Play): Iter {cfg.stage1_iterations+1} a {cfg.total_iterations}")


In [ ]:
# 5. Inicialização da Rede Neural
import os, sys, torch
import torch.optim as optim
from haxball.rl.models.mlp_policy import ActorCriticMLP
from haxball.rl.models.entity_attention import EntityAttentionPolicy

def create_agent(cfg, device, warm_start_checkpoint=None):
    if getattr(cfg, 'policy_type', 'mlp') == 'attention':
        agent = EntityAttentionPolicy(embed_dim=64, num_heads=4, act_dim=cfg.num_actions, is_discrete=True)
    else:
        agent = ActorCriticMLP(obs_dim=cfg.obs_dim, act_dim=cfg.num_actions, is_discrete=True, hidden_dim=128)
    
    if warm_start_checkpoint and os.path.exists(warm_start_checkpoint):
        try:
            state = torch.load(warm_start_checkpoint, map_location=device)
            agent.load_state_dict(state, strict=False)
            print(f'✓ Warm-start carregado de: {warm_start_checkpoint}')
        except Exception as e:
            print(f'⚠️ Iniciando do ZERO (Tabula Rasa). Motivo: {e}')
    else:
        print('✓ Agente inicializado do ZERO (Tabula Rasa) com pesos ortogonais!')
    
    agent = agent.to(device)
    return agent

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
warm_path = 'checkpoints/human_expert_bc.pt' if getattr(cfg, 'WARM_START', False) or (globals().get('WARM_START', False)) else None
agent = create_agent(cfg, device, warm_start_checkpoint=warm_path)
optimizer = optim.Adam(agent.parameters(), lr=cfg.lr, eps=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.total_iterations, eta_min=1e-5)


In [ ]:
# 5. Inicialização da Rede Neural
def create_agent(cfg, device, warm_start_checkpoint=None):
    if cfg.policy_type == 'attention':
        agent = EntityAttentionPolicy(embed_dim=64, num_heads=4, act_dim=cfg.num_actions, is_discrete=True).to(device)
    else:
        agent = ActorCriticMLP(obs_dim=cfg.obs_dim, act_dim=cfg.num_actions, is_discrete=True, hidden_dim=128).to(device)
    
    if warm_start_checkpoint and os.path.exists(warm_start_checkpoint):
        try:
            state = torch.load(warm_start_checkpoint, map_location=device)
            agent.load_state_dict(state, strict=False)
            print(f'✓ Warm-start carregado de: {warm_start_checkpoint}')
        except Exception as e:
            print(f'⚠️ Iniciando do ZERO (Tabula Rasa). Motivo: {e}')
    else:
        print('✓ Agente inicializado do ZERO (Tabula Rasa) com pesos ortogonais!')
    
    return agent

warm_path = 'checkpoints/human_expert_bc.pt' if WARM_START else None
agent = create_agent(cfg, device, warm_start_checkpoint=warm_path)
optimizer = optim.Adam(agent.parameters(), lr=cfg.lr, eps=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.total_iterations, eta_min=1e-5)


In [ ]:
# 6. Gauntlet de Avaliação e Benchmark Rápido
def evaluate_agent_gauntlet(agent, stadium_file, num_matches=2):
    test_opponents = [
        ('Heuristic', HeuristicBot(name='Heuristic')),
        ('Striker', StrikerBot(name='Striker')),
        ('WallRebound', WallReboundBot(name='WallRebound')),
        ('Master', MasterBot(name='Master'))
    ]
    
    results = {}
    for name, bot in test_opponents:
        env = HaxBallEnv(stadium_file=stadium_file, opponent_bot=bot, max_steps=1200)
        wins, goals_for, goals_against = 0, 0, 0
        
        for _ in range(num_matches):
            obs, _ = env.reset()
            done = False
            while not done:
                obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    action_idx, _, _, _ = agent.get_action_and_value(obs_t)
                act_val = int(action_idx.cpu().item())
                obs, _, terminated, truncated, info = env.step(act_val)
                done = terminated or truncated
            
            if env.game.red_score > env.game.blue_score:
                wins += 1
            goals_for += env.game.red_score
            goals_against += env.game.blue_score
            
        win_pct = (wins / num_matches) * 100.0
        results[name] = {'win_rate': win_pct, 'gf': goals_for, 'ga': goals_against}
        
    return results


In [ ]:
# 7. Loop Principal de Treinamento PPO (Self-Play League Pool + Gráficos ao Vivo)
metrics = {
    'iteration': [],
    'total_steps': [],
    'mean_reward': [],
    'win_rate': [],
    'policy_loss': [],
    'value_loss': [],
    'entropy': [],
    'gauntlet_score': []
}

league_pool = []
opp_bot = HeuristicBot(name='Heuristic_Coach')
env = HaxBallEnv(stadium_file=cfg.stadium_file, opponent_bot=opp_bot, max_steps=1500)
best_gauntlet_score = -1.0
start_time = time.time()

print('🚀 INICIANDO TREINAMENTO REFORÇO (PPO + SELF-PLAY LEAGUE)...')

for iteration in range(1, cfg.total_iterations + 1):
    # Gestão da Liga de Self-Play
    if iteration == cfg.stage1_iterations + 1:
        print('\n⭐ TRANSIÇÃO PARA FASE 2: LIGA DE AUTO-CONFRONTO (SELF-PLAY SIMÉTRICO)!')
        gen_path = os.path.join(cfg.checkpoint_dir, 'gen_stage1.pt')
        torch.save(agent.state_dict(), gen_path)
        league_pool.append(gen_path)
        
    # Atualiza pool de oponentes a cada 50 iterações na Fase 2
    if iteration > cfg.stage1_iterations and iteration % 50 == 0:
        gen_path = os.path.join(cfg.checkpoint_dir, f'gen_iter_{iteration}.pt')
        torch.save(agent.state_dict(), gen_path)
        league_pool.append(gen_path)
        
    # Sorteia oponente da rodada (70% versão recente, 20% checkpoint histórico, 10% heurístico)
    if iteration > cfg.stage1_iterations and len(league_pool) > 0:
        p_choice = random.random()
        if p_choice < 0.70:
            chosen_model = league_pool[-1]  # Versão mais recente
            opp_bot = RLBot(model_path=chosen_model, name='Recent_Self')
        elif p_choice < 0.90:
            chosen_model = random.choice(league_pool) # Checkpoint histórico
            opp_bot = RLBot(model_path=chosen_model, name='Historic_Self')
        else:
            opp_bot = PressBot(name='Press_Baseline') # Fundamentos de marcação
            
        env.opponent_bot = opp_bot
    
    # Coleta de Rollouts
    obs_buf = torch.zeros((cfg.num_steps, cfg.obs_dim), device=device)
    act_buf = torch.zeros(cfg.num_steps, dtype=torch.long, device=device)
    logprob_buf = torch.zeros(cfg.num_steps, device=device)
    rew_buf = torch.zeros(cfg.num_steps, device=device)
    done_buf = torch.zeros(cfg.num_steps, device=device)
    val_buf = torch.zeros(cfg.num_steps, device=device)
    
    obs, _ = env.reset()
    obs_t = torch.tensor(obs, dtype=torch.float32, device=device)
    
    ep_rewards, ep_wins, ep_count, cur_rew = [], 0, 0, 0.0
    
    for step in range(cfg.num_steps):
        obs_buf[step] = obs_t
        with torch.no_grad():
            action, logprob, _, value = agent.get_action_and_value(obs_t.unsqueeze(0))
            val_buf[step] = value.flatten()
        
        act_buf[step] = action.squeeze()
        logprob_buf[step] = logprob.squeeze()
        
        act_val = int(action.cpu().item())
        next_obs, reward, term, trunc, info = env.step(act_val)
        done = term or trunc
        
        rew_buf[step] = reward
        done_buf[step] = float(done)
        cur_rew += reward
        
        if done:
            ep_rewards.append(cur_rew)
            ep_count += 1
            if env.game.red_score > env.game.blue_score:
                ep_wins += 1
            cur_rew = 0.0
            next_obs, _ = env.reset()
        
        obs_t = torch.tensor(next_obs, dtype=torch.float32, device=device)
    
    # Cálculo de Vantagem GAE
    with torch.no_grad():
        next_val = agent.get_value(obs_t.unsqueeze(0)).reshape(1, -1)
        advantages = torch.zeros_like(rew_buf, device=device)
        lastgaelam = 0.0
        for t in reversed(range(cfg.num_steps)):
            if t == cfg.num_steps - 1:
                nextnonterminal = 1.0 - done_buf[t]
                nextvalues = next_val
            else:
                nextnonterminal = 1.0 - done_buf[t]
                nextvalues = val_buf[t + 1]
            delta = rew_buf[t] + cfg.gamma * nextvalues * nextnonterminal - val_buf[t]
            advantages[t] = lastgaelam = delta + cfg.gamma * cfg.gae_lambda * nextnonterminal * lastgaelam
        returns = advantages + val_buf
    
    # Atualização PPO com Minibatches
    b_inds = np.arange(cfg.num_steps)
    tot_pg_loss, tot_v_loss, tot_ent, n_updates = 0.0, 0.0, 0.0, 0
    
    for epoch in range(cfg.update_epochs):
        np.random.shuffle(b_inds)
        for start in range(0, cfg.num_steps, cfg.batch_size):
            mb_inds = b_inds[start:start + cfg.batch_size]
            
            _, newlogprob, entropy, newval = agent.get_action_and_value(obs_buf[mb_inds], act_buf[mb_inds])
            logratio = newlogprob - logprob_buf[mb_inds]
            ratio = logratio.exp()
            
            mb_adv = advantages[mb_inds]
            mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)
            
            pg_loss1 = -mb_adv * ratio
            pg_loss2 = -mb_adv * torch.clamp(ratio, 1 - cfg.clip_coef, 1 + cfg.clip_coef)
            pg_loss = torch.max(pg_loss1, pg_loss2).mean()
            
            v_loss = 0.5 * ((newval.view(-1) - returns[mb_inds]) ** 2).mean()
            ent_loss = entropy.mean()
            loss = pg_loss - cfg.ent_coef * ent_loss + cfg.vf_coef * v_loss
            
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), cfg.max_grad_norm)
            optimizer.step()
            
            tot_pg_loss += pg_loss.item()
            tot_v_loss += v_loss.item()
            tot_ent += ent_loss.item()
            n_updates += 1
            
    scheduler.step()
    
    # Registro de Métricas
    mean_rew = float(np.mean(ep_rewards)) if ep_rewards else 0.0
    win_rt = (ep_wins / ep_count * 100.0) if ep_count else 0.0
    
    metrics['iteration'].append(iteration)
    metrics['total_steps'].append(iteration * cfg.num_steps)
    metrics['mean_reward'].append(mean_rew)
    metrics['win_rate'].append(win_rt)
    metrics['policy_loss'].append(tot_pg_loss / max(1, n_updates))
    metrics['value_loss'].append(tot_v_loss / max(1, n_updates))
    metrics['entropy'].append(tot_ent / max(1, n_updates))
    
    # Avaliação Periódica no Gauntlet
    gauntlet_avg = 0.0
    if iteration % cfg.eval_frequency == 0 or iteration == 1:
        g_res = evaluate_agent_gauntlet(agent, cfg.stadium_file, num_matches=2)
        gauntlet_avg = float(np.mean([v['win_rate'] for v in g_res.values()]))
        metrics['gauntlet_score'].append(gauntlet_avg)
        
        if gauntlet_avg >= best_gauntlet_score:
            best_gauntlet_score = gauntlet_avg
            best_path = os.path.join(cfg.checkpoint_dir, 'haxball_rl_best.pt')
            torch.save(agent.state_dict(), best_path)
            
        clear_output(wait=True)
        fig, axs = plt.subplots(2, 2, figsize=(14, 8))
        fig.suptitle(f'⚽ HaxBall PPO Reinforcement Learning | Iter {iteration}/{cfg.total_iterations} ({iteration*cfg.num_steps:,} Passos)', fontsize=14, fontweight='bold')
        
        axs[0, 0].plot(metrics['iteration'], metrics['mean_reward'], color='#2ecc71', lw=2)
        axs[0, 0].set_title('Recompensa Média por Episódio')
        axs[0, 0].set_xlabel('Iteração')
        axs[0, 0].grid(True, alpha=0.3)
        
        axs[0, 1].plot(metrics['iteration'], metrics['win_rate'], color='#3498db', lw=2, label='Win Rate Treino')
        axs[0, 1].set_title('Taxa de Vitória no Treino (%)')
        axs[0, 1].set_xlabel('Iteração')
        axs[0, 1].grid(True, alpha=0.3)
        
        axs[1, 0].plot(metrics['iteration'], metrics['policy_loss'], color='#e74c3c', lw=2, label='Policy Loss')
        axs[1, 0].plot(metrics['iteration'], metrics['value_loss'], color='#f39c12', lw=1.5, label='Value Loss')
        axs[1, 0].set_title('Função de Perda (Loss)')
        axs[1, 0].set_xlabel('Iteração')
        axs[1, 0].legend()
        axs[1, 0].grid(True, alpha=0.3)
        
        axs[1, 1].plot(metrics['iteration'], metrics['entropy'], color='#9b59b6', lw=2)
        axs[1, 1].set_title('Entropia da Política (Exploração)')
        axs[1, 1].set_xlabel('Iteração')
        axs[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        display(fig)
        plt.close(fig)
        
        print(f'\n[Iter {iteration:04d}/{cfg.total_iterations}] Reward: {mean_rew:+.2f} | WinRate: {win_rt:.1f}% | Gauntlet Score: {gauntlet_avg:.1f}% | Recorde Gauntlet: {best_gauntlet_score:.1f}%')
        for opp_n, d in g_res.items():
            print(f"  vs {opp_n:12s}: WinRate {d['win_rate']:5.1f}% | Gols Pró: {d['gf']} | Gols Contra: {d['ga']}")

final_path = os.path.join(cfg.checkpoint_dir, 'haxball_rl_final.pt')
torch.save(agent.state_dict(), final_path)
elapsed = time.time() - start_time
print(f'\n✅ TREINAMENTO CONCLUÍDO COM SUCESSO EM {elapsed/60:.2f} MINUTOS!')


In [ ]:
# @title 8. 🏆 COPA HAXBALL AI: Grande Benchmark Final de Modelos & Liga de Bots
from tabulate import tabulate

print('========================================================================')
print('🏅 EXECUTANDO TORNEIO FINAL DE AVALIAÇÃO (GRAND BENCHMARK MATRIX)...')
print('========================================================================\n')

# 1. Torneio Completo contra TODOS os 8 Bots Scripts
all_bots = [
    ('Heuristic (Básico)', HeuristicBot(name='Heuristic')),
    ('Striker (Artilheiro)', StrikerBot(name='Striker')),
    ('Counter (Muralha)', CounterBot(name='Counter')),
    ('WallRebound (Tabelador)', WallReboundBot(name='Wall')),
    ('PressBot (Pressão Total)', PressBot(name='Press')),
    ('Dribbler (Fintador)', DribblerBot(name='Dribbler')),
    ('BankBot (Bate na Parede)', BankBot(name='Bank')),
    ('MasterBot (Mestre Supremo)', MasterBot(name='Master')),
]

best_bot = RLBot(policy=agent, name='RL_Champion')
table_data = []
stad = Stadium.load_from_file(cfg.stadium_file)

for b_name, b_inst in all_bots:
    wins, gf, ga, shots, touches = 0, 0, 0, 0, 0
    n_test_matches = 5
    
    for _ in range(n_test_matches):
        game = HaxBallGame(stadium=stad, score_limit=3, time_limit_secs=120)
        p_red = [p for p in game.players if p.team == Team.RED][0]
        p_blue = [p for p in game.players if p.team == Team.BLUE][0]
        
        step_count = 0
        while game.state.name != 'GAME_OVER' and step_count < 3000:
            cmd_r = best_bot.act(game, p_red)
            cmd_b = b_inst.act(game, p_blue)
            
            if cmd_r[2]: shots += 1
            info = game.step({p_red.player_id: cmd_r, p_blue.player_id: cmd_b})
            for c in info['events'].get('disc_ball_collisions', []):
                if c.get('player_id') == p_red.player_id: touches += 1
            step_count += 1
            
        if game.red_score > game.blue_score: wins += 1
        gf += game.red_score
        ga += game.blue_score
        
    win_pct = (wins / n_test_matches) * 100.0
    diff = gf - ga
    table_data.append([b_name, f'{win_pct:.1f}%', f'{wins}/{n_test_matches}', f'{gf} - {ga}', f'{diff:+d}', touches, shots])

headers = ['Adversário', 'Taxa de Vitória', 'V-D', 'Gols (Pró-Contra)', 'Saldo', 'Toques', 'Chutes']
print(tabulate(table_data, headers=headers, tablefmt='fancy_grid'))

# 2. Confronto Round-Robin entre Gerações da IA (se houver mais de 1 geração salva)
if len(league_pool) > 1:
    print('\n========================================================================')
    print('⚔️ CONFRONTO HISTÓRICO ENTRE GERAÇÕES DA IA (ROUND-ROBIN MATRIX)')
    print('========================================================================')
    gens_to_test = league_pool[-4:] if len(league_pool) >= 4 else league_pool
    gen_names = [os.path.basename(g).replace('.pt', '') for g in gens_to_test] + ['Versão_Final']
    gen_bots = [RLBot(model_path=g) for g in gens_to_test] + [best_bot]
    
    rr_matrix = []
    for i, bot_a in enumerate(gen_bots):
        row = [gen_names[i]]
        for j, bot_b in enumerate(gen_bots):
            if i == j:
                row.append('-')
            else:
                game = HaxBallGame(stadium=stad, score_limit=3, time_limit_secs=90)
                p_r = [p for p in game.players if p.team == Team.RED][0]
                p_b = [p for p in game.players if p.team == Team.BLUE][0]
                for _ in range(2500):
                    if game.state.name == 'GAME_OVER': break
                    cmd_a = bot_a.act(game, p_r)
                    cmd_b = bot_b.act(game, p_b)
                    game.step({p_r.player_id: cmd_a, p_b.player_id: cmd_b})
                row.append(f'{game.red_score}x{game.blue_score}')
        rr_matrix.append(row)
        
    print(tabulate(rr_matrix, headers=['Geração'] + gen_names, tablefmt='grid'))


In [ ]:
# 9. Download do Modelo Campeão
try:
    from google.colab import files
    print('📥 Baixando o melhor modelo treinado para sua máquina local...')
    best_model_file = 'checkpoints/haxball_rl_best.pt' if os.path.exists('checkpoints/haxball_rl_best.pt') else 'checkpoints/haxball_rl_final.pt'
    files.download(best_model_file)
except Exception:
    print('Execute no Colab para baixar os pesos automaticamente.')
